# Stock Forecasting: ANN vs SVM Comparative Analysis

This notebook demonstrates a comprehensive comparison between Artificial Neural Networks (ANN) and Support Vector Machines (SVM) for stock price forecasting using the VN30 dataset.

## Notebook Overview
1. Data loading and exploration
2. Feature engineering with technical indicators
3. ANN model implementation
4. SVM model implementation
5. Model evaluation and comparison
6. Visualization of results

**Dataset**: VN30 Vietnamese Stock Market (2015-2026)

**Models Compared**:
- Dense ANN (Feedforward Neural Network)
- LSTM (Long Short-Term Memory)
- SVM (Support Vector Regression)

## 1. Import Required Libraries

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
from data_preprocessing import StockDataPreprocessor
from ann_models import DenseANN, LSTMModel
from svm_models import SVMModel, SVMEnsemble
from evaluation import ModelEvaluator
import config

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 6)

print("All libraries imported successfully!")
print(f"Dataset path: {config.DATA_PATH}")

## 2. Load and Explore the Dataset

Let's load the VN30 dataset and perform initial exploration.

In [ ]:
# Load the dataset
df = pd.read_csv(config.DATA_PATH)
df['time'] = pd.to_datetime(df['time'])

# Display basic information
print("Dataset Shape:", df.shape)
print("\nDataset Info:")
print(df.info())
print("\nFirst few rows:")
df.head()

In [ ]:
# Explore available tickers
tickers = df['Ticker'].unique()
print(f"Number of unique tickers: {len(tickers)}")
print(f"\nAvailable tickers:\n{tickers}")

# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")

# Statistical summary
print("\nStatistical Summary:")
df.describe()

In [ ]:
# Select a specific ticker for analysis (you can change this)
SELECTED_TICKER = tickers[0]  # Using first ticker
print(f"Selected ticker for analysis: {SELECTED_TICKER}")

# Filter data for selected ticker
stock_data = df[df['Ticker'] == SELECTED_TICKER].copy()
stock_data = stock_data.sort_values('time').reset_index(drop=True)

print(f"\nNumber of records for {SELECTED_TICKER}: {len(stock_data)}")
print(f"Date range: {stock_data['time'].min()} to {stock_data['time'].max()}")

# Plot closing price over time
plt.figure(figsize=(15, 5))
plt.plot(stock_data['time'], stock_data['close'], linewidth=1)
plt.title(f'{SELECTED_TICKER} - Closing Price Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Data Preprocessing and Feature Engineering

Now we'll use our preprocessing module to add technical indicators and prepare the data.

In [ ]:
# Initialize preprocessor for ANN (no sequences - for Dense ANN and SVM)
print("="*70)
print("PREPROCESSING DATA FOR DENSE ANN AND SVM")
print("="*70)

preprocessor_ann = StockDataPreprocessor()
ann_data = preprocessor_ann.preprocess_for_ann(ticker=SELECTED_TICKER, use_sequences=False)

print(f"\nFeature shape: {ann_data['X_train'].shape}")
print(f"Number of features: {len(ann_data['feature_columns'])}")
print(f"\nFeatures used:")
for i, feature in enumerate(ann_data['feature_columns'], 1):
    print(f"  {i}. {feature}")

In [ ]:
# Initialize preprocessor for LSTM (with sequences)
print("\n" + "="*70)
print("PREPROCESSING DATA FOR LSTM")
print("="*70)

preprocessor_lstm = StockDataPreprocessor()
lstm_data = preprocessor_lstm.preprocess_for_ann(ticker=SELECTED_TICKER, use_sequences=True)

print(f"\nLSTM sequence shape: {lstm_data['X_train'].shape}")
print(f"  - Samples: {lstm_data['X_train'].shape[0]}")
print(f"  - Time steps: {lstm_data['X_train'].shape[1]}")
print(f"  - Features per step: {lstm_data['X_train'].shape[2]}")

## 4. Implement and Train Dense ANN Model

Let's train a Dense (Feedforward) Neural Network for stock prediction.

In [ ]:
# Build and display Dense ANN architecture
input_shape = (ann_data['X_train'].shape[1],)
dense_ann = DenseANN(input_shape)
dense_ann.build_model()

print("Dense ANN Architecture:")
dense_ann.get_model_summary()

In [ ]:
# Train Dense ANN
print("Training Dense ANN...")
history_ann = dense_ann.train(
    ann_data['X_train'], 
    ann_data['y_train'],
    ann_data['X_val'],
    ann_data['y_val'],
    verbose=1
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_ann.history['loss'], label='Training Loss')
axes[0].plot(history_ann.history['val_loss'], label='Validation Loss')
axes[0].set_title('Dense ANN - Loss Over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_ann.history['mae'], label='Training MAE')
axes[1].plot(history_ann.history['val_mae'], label='Validation MAE')
axes[1].set_title('Dense ANN - MAE Over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Implement and Train LSTM Model

LSTM can capture temporal dependencies in sequential stock data.

In [ ]:
# Build and display LSTM architecture
input_shape_lstm = (lstm_data['X_train'].shape[1], lstm_data['X_train'].shape[2])
lstm_model = LSTMModel(input_shape_lstm)
lstm_model.build_model()

print("LSTM Architecture:")
lstm_model.get_model_summary()

In [ ]:
# Train LSTM
print("Training LSTM...")
history_lstm = lstm_model.train(
    lstm_data['X_train'],
    lstm_data['y_train'],
    lstm_data['X_val'],
    lstm_data['y_val'],
    verbose=1
)

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(history_lstm.history['loss'], label='Training Loss')
axes[0].plot(history_lstm.history['val_loss'], label='Validation Loss')
axes[0].set_title('LSTM - Loss Over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_lstm.history['mae'], label='Training MAE')
axes[1].plot(history_lstm.history['val_mae'], label='Validation MAE')
axes[1].set_title('LSTM - MAE Over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Implement and Train SVM Model

Support Vector Regression for stock price prediction.

In [ ]:
# Prepare SVM data (same as Dense ANN data)
preprocessor_svm = StockDataPreprocessor()
svm_data = preprocessor_svm.preprocess_for_svm(ticker=SELECTED_TICKER)

# Build and train SVM with RBF kernel
print("Training SVM with RBF kernel...")
svm_model = SVMModel(kernel='rbf', C=config.SVM_CONFIG['C'], 
                     epsilon=config.SVM_CONFIG['epsilon'], 
                     gamma=config.SVM_CONFIG['gamma'])
svm_model.train(svm_data['X_train'], svm_data['y_train'], verbose=1)

print("\nSVM Model Info:")
print(svm_model.get_model_info())

## 7. Make Predictions with All Models

Generate predictions from all trained models on the test set.

In [ ]:
# Dense ANN predictions
y_pred_ann_scaled = dense_ann.predict(ann_data['X_test']).flatten()
y_pred_ann = preprocessor_ann.inverse_transform_target(y_pred_ann_scaled)
y_true_ann = preprocessor_ann.inverse_transform_target(ann_data['y_test'])

print("Dense ANN predictions generated")

# LSTM predictions
y_pred_lstm_scaled = lstm_model.predict(lstm_data['X_test']).flatten()
y_pred_lstm = preprocessor_lstm.inverse_transform_target(y_pred_lstm_scaled)
y_true_lstm = preprocessor_lstm.inverse_transform_target(lstm_data['y_test'])

print("LSTM predictions generated")

# SVM predictions
y_pred_svm_scaled = svm_model.predict(svm_data['X_test'])
y_pred_svm = preprocessor_svm.inverse_transform_target(y_pred_svm_scaled)
y_true_svm = preprocessor_svm.inverse_transform_target(svm_data['y_test'])

print("SVM predictions generated")
print("\nAll predictions complete!")

## 8. Model Evaluation and Comparison

Evaluate all models using comprehensive metrics.

In [ ]:
# Create evaluator
evaluator = ModelEvaluator()

# Evaluate each model
print("="*70)
print("EVALUATING ALL MODELS")
print("="*70)

# Use the same true values (they should be nearly identical, use one as reference)
y_true = y_true_ann  # Reference true values

# Create predictions dictionary
predictions_dict = {
    'Dense_ANN': y_pred_ann,
    'LSTM': y_pred_lstm,
    'SVM_RBF': y_pred_svm
}

# Evaluate each model
for model_name, y_pred in predictions_dict.items():
    evaluator.evaluate_model(model_name, y_true, y_pred)
    evaluator.print_evaluation(model_name)

In [ ]:
# Compare all models
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70 + "\n")

comparison_df = evaluator.compare_models()
comparison_df

## 9. Visualize Predictions vs Actual Values

Compare predictions from all models against actual stock prices.

In [ ]:
# Plot predictions comparison
n_samples = 200  # Number of recent samples to plot

plt.figure(figsize=(16, 6))

x = np.arange(len(y_true[-n_samples:]))
plt.plot(x, y_true[-n_samples:], 'k-', label='Actual', linewidth=2.5, alpha=0.8)
plt.plot(x, y_pred_ann[-n_samples:], '--', label='Dense ANN', linewidth=1.5, alpha=0.7)
plt.plot(x, y_pred_lstm[-n_samples:], '--', label='LSTM', linewidth=1.5, alpha=0.7)
plt.plot(x, y_pred_svm[-n_samples:], '--', label='SVM (RBF)', linewidth=1.5, alpha=0.7)

plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('Stock Price', fontsize=12)
plt.title(f'{SELECTED_TICKER} - Actual vs Predicted Stock Prices (Last {n_samples} samples)', 
          fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: Actual vs Predicted
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [('Dense ANN', y_pred_ann), ('LSTM', y_pred_lstm), ('SVM (RBF)', y_pred_svm)]

for ax, (model_name, y_pred) in zip(axes, models):
    ax.scatter(y_true, y_pred, alpha=0.5, s=10)
    
    # Plot perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    # Calculate R²
    from sklearn.metrics import r2_score
    r2 = r2_score(y_true, y_pred)
    
    ax.set_xlabel('Actual Values', fontsize=11)
    ax.set_ylabel('Predicted Values', fontsize=11)
    ax.set_title(f'{model_name}\nR² = {r2:.4f}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Performance Metrics Analysis

Visualize and analyze the performance metrics for all models.

In [ ]:
# Bar charts for metrics comparison
metrics_to_plot = ['MAE', 'RMSE', 'MAPE', 'R2', 'DA']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    
    values = comparison_df[metric].values
    models_list = comparison_df.index.tolist()
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
    bars = ax.bar(models_list, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Highlight best model
    if metric in ['MAE', 'RMSE', 'MAPE']:
        best_idx = np.argmin(values)
        label_text = 'Lower is Better'
    else:
        best_idx = np.argmax(values)
        label_text = 'Higher is Better'
    
    bars[best_idx].set_edgecolor('red')
    bars[best_idx].set_linewidth(3)
    
    ax.set_ylabel(metric, fontsize=12, fontweight='bold')
    ax.set_title(f'{metric} Comparison\n({label_text})', fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}',
               ha='center', va='bottom', fontsize=10, fontweight='bold')

# Remove extra subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.show()

In [ ]:
# Error distribution for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_name, y_pred) in zip(axes, [('Dense ANN', y_pred_ann), 
                                             ('LSTM', y_pred_lstm), 
                                             ('SVM (RBF)', y_pred_svm)]):
    errors = y_true - y_pred
    
    ax.hist(errors, bins=50, alpha=0.7, edgecolor='black', color='steelblue')
    ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    ax.axvline(np.mean(errors), color='green', linestyle='--', linewidth=2, 
               label=f'Mean: {np.mean(errors):.4f}')
    
    ax.set_xlabel('Prediction Error', fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{model_name} - Error Distribution\nStd: {np.std(errors):.4f}', 
                fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Conclusions and Insights

### Key Findings

Based on the comprehensive evaluation of ANN and SVM models for stock forecasting:

**Model Performance:**
- Compare the models based on multiple metrics (MAE, RMSE, MAPE, R², DA)
- Identify which model performs best for this specific stock
- Analyze the trade-offs between different approaches

**Model Characteristics:**

**Dense ANN:**
- ✅ Can capture complex non-linear relationships
- ✅ Flexible architecture
- ⚠️ Requires careful tuning to avoid overfitting
- ⚠️ Longer training time

**LSTM:**
- ✅ Captures temporal dependencies and sequences
- ✅ Good for time-series data
- ⚠️ More complex and computationally expensive
- ⚠️ Requires more data for optimal performance

**SVM:**
- ✅ Robust to outliers
- ✅ Works well with smaller datasets
- ✅ Faster training compared to deep learning
- ⚠️ Sensitive to kernel and hyperparameter selection
- ⚠️ May not capture complex temporal patterns

### Recommendations

1. **Ensemble Approach**: Combine predictions from multiple models for more robust forecasts
2. **Feature Engineering**: Technical indicators significantly impact performance
3. **Hyperparameter Tuning**: Use grid search or Bayesian optimization for better results
4. **Cross-Validation**: Implement time-series cross-validation for more reliable estimates
5. **Regular Retraining**: Update models with new data to maintain accuracy

## 12. Next Steps & Future Improvements

### Suggested Improvements for Your Project:

#### 1. **Advanced Models**
```python
# Add more sophisticated models:
- Transformer-based models (Attention mechanisms)
- GRU (Gated Recurrent Unit)
- Hybrid models (CNN-LSTM)
- Ensemble methods (Stacking, Blending)
```

#### 2. **Enhanced Feature Engineering**
```python
# Additional features to consider:
- Fundamental analysis (P/E ratio, EPS, etc.)
- Market sentiment from news/social media
- Macroeconomic indicators (interest rates, GDP, etc.)
- Cross-stock correlations
- Market volatility index (VIX equivalent)
```

#### 3. **Advanced Techniques**
- **Multi-step Forecasting**: Predict multiple days ahead
- **Transfer Learning**: Train on one stock, apply to others
- **Attention Mechanisms**: Identify which features matter most
- **Explainable AI**: Use SHAP/LIME to understand predictions

#### 4. **Optimization & Tuning**
- Implement Bayesian optimization for hyperparameters
- Use AutoML tools (Auto-Keras, Optuna)
- Neural Architecture Search (NAS)
- Implement early stopping with learning rate scheduling

#### 5. **Backtesting & Trading Strategy**
- Build a simulated trading system
- Calculate Sharpe ratio, max drawdown
- Test different entry/exit strategies
- Risk management rules

#### 6. **Real-time Pipeline**
- Connect to live data feeds
- Implement online learning
- Create REST API for predictions
- Build web dashboard (Streamlit/Dash)

#### 7. **Robustness Testing**
- Cross-validation across different time periods
- Test on multiple stocks
- Stress testing with market crashes
- Monte Carlo simulations

#### 8. **Documentation & Deployment**
- Write technical report with findings
- Create presentation of results
- Deploy model as web service
- Set up monitoring and alerting

### Quick Experiments You Can Try Now:

1. **Change the ticker**: Modify `SELECTED_TICKER` to analyze different stocks
2. **Adjust sequence length**: Change `config.SEQUENCE_LENGTH` for LSTM
3. **Try different SVM kernels**: 'linear', 'poly', 'sigmoid'
4. **Modify ANN architecture**: Add/remove layers in `config.py`
5. **Experiment with technical indicators**: Add more indicators in config

### Resources for Learning More:

- **Papers**: Google Scholar for "stock prediction deep learning"
- **Books**: "Advances in Financial Machine Learning" by Marcos López de Prado
- **Online Courses**: Coursera, Udacity for Time Series Forecasting
- **Communities**: Kaggle competitions, QuantConnect forums

In [ ]:
# Save all models for future use
print("Saving trained models...")
dense_ann.save_model()
lstm_model.save_model()
svm_model.save_model()
print("\n✅ All models saved successfully!")
print(f"\nModels saved to: {config.MODEL_DIR}/")
print(f"Results saved to: {config.OUTPUT_DIR}/")
print("\n" + "="*70)
print("ANALYSIS COMPLETE!")
print("="*70)